In [3]:
import pandas as pd

df_marking = pd.read_csv('df_with_marking_final.csv')

In [4]:
df_marking["time_of_day"].value_counts(dropna=False)

time_of_day
вечер                33
ночь                 24
день                 13
утро                 12
неизвестно            6
['ночь', 'утро']      5
['утро', 'день']      3
['вечер', 'ночь']     2
['день', 'вечер']     2
Name: count, dtype: int64

In [5]:
import re
import pandas as pd

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

time_keywords = {
    'утро': ['утром', 'утра', 'ранним утром', 'с утра', 'к утру'],
    'день': ['днем', 'дня', 'в полдень', 'к обеду'],
    'вечер': ['вечером', 'вечера', 'поздним вечером', 'к вечеру'],
    'ночь': ['ночью', 'ночи', 'глубокой ночью', 'за полночь']
}

train_data_time_of_day = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()

    true_label = row.get("time_of_day")
    if pd.isnull(true_label) or true_label not in time_keywords:
        continue

    found = False
    for keyword in time_keywords[true_label]:
        match = re.search(r'\b' + re.escape(keyword) + r'\b', text_lower)
        if match:
            start, end = match.span()
            print(text[start:end])
            train_data_time_of_day.append((text, {"entities": [(start, end, "TIME_OF_DAY")]}))
            found = True
            break

    if not found:
        train_data_time_of_day.append((text, {"entities": []}))
        print(f"Не найдены ключевые слова '{true_label}' в id={row['id']}")

print(f"TRAIN_DATA_TIME_OF_DAY готово: {len(train_data_time_of_day)} примеров")

Не найдены ключевые слова 'ночь' в id=71745
вечером
Не найдены ключевые слова 'утро' в id=102232
Не найдены ключевые слова 'ночь' в id=87680
Вечером
вечером
Не найдены ключевые слова 'вечер' в id=12555
Вечером
вечером
Утром
днем
ночью
утром
ночи
ночью
утром
дня
вечером
ночью
ночью
Не найдены ключевые слова 'ночь' в id=71223
Ночью
Ночью
дня
ночью
Не найдены ключевые слова 'вечер' в id=13573
вечером
Не найдены ключевые слова 'ночь' в id=113143
Не найдены ключевые слова 'ночь' в id=42500
Не найдены ключевые слова 'ночь' в id=124977
Ночью
ночи
Не найдены ключевые слова 'утро' в id=105000
Не найдены ключевые слова 'ночь' в id=69674
Не найдены ключевые слова 'ночь' в id=66092
ночи
Не найдены ключевые слова 'ночь' в id=113151
Не найдены ключевые слова 'вечер' в id=58548
дня
Не найдены ключевые слова 'вечер' в id=112892
Не найдены ключевые слова 'вечер' в id=32496
утром
Утром
днем
Не найдены ключевые слова 'вечер' в id=91014
вечером
Вечером
Не найдены ключевые слова 'вечер' в id=89433
Не найде

In [6]:
print(train_data_time_of_day[1][1])
train_data_time_of_day[1][0][6392:6399]

{'entities': [(6392, 6399, 'TIME_OF_DAY')]}


'вечером'

In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch
import random
import warnings

nlp = spacy.blank("ru")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

ner.add_label("TIME_OF_DAY")

examples = []
for text, annot in train_data_time_of_day:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annot)
    examples.append(example)

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(15):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

nlp.to_disk("ner_time_of_day_model")
print("Модель сохранена в 'ner_time_of_day_model'")

Epoch 1, Losses: {'ner': 262120.80478879157}
Epoch 2, Losses: {'ner': 112.13895587480833}
Epoch 3, Losses: {'ner': 111.75436051718526}
Epoch 4, Losses: {'ner': 110.94516814916553}
Epoch 5, Losses: {'ner': 94.17841744993684}
Epoch 6, Losses: {'ner': 88.40301964058607}
Epoch 7, Losses: {'ner': 84.22824007839901}
Epoch 8, Losses: {'ner': 74.55751643116379}
Epoch 9, Losses: {'ner': 69.51898479974336}
Epoch 10, Losses: {'ner': 60.040344873965886}
Epoch 11, Losses: {'ner': 53.907034088355516}
Epoch 12, Losses: {'ner': 45.5756015026116}
Epoch 13, Losses: {'ner': 40.06991249212988}
Epoch 14, Losses: {'ner': 30.249209294345277}
Epoch 15, Losses: {'ner': 39.97145416051442}
✅ Модель сохранена в 'ner_time_of_day_model'


In [25]:
for i in range(15, 20):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

Epoch 16, Losses: {'ner': 38.75949885571107}
Epoch 17, Losses: {'ner': 28.66716471791031}
Epoch 18, Losses: {'ner': 26.376371077319625}
Epoch 19, Losses: {'ner': 26.642533809265526}
Epoch 20, Losses: {'ner': 21.079010882407967}


In [ ]:
for i in range(16, 20):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

In [ ]:
nlp.to_disk("ner_time_of_day_model")
print("Модель сохранена в 'ner_time_of_day_model'")

Модель сохранена в 'ner_time_of_day_model'


In [7]:
import spacy
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

nlp = spacy.load("ner_time_of_day_model")

time_mapping = {'утро': 0, 'день': 1, 'вечер': 2, 'ночь': 3}
time_keywords = {
    'утро': ['утром', 'утра', 'ранним утром', 'с утра', 'к утру'],
    'день': ['днем', 'дня', 'в полдень', 'к обеду'],
    'вечер': ['вечером', 'вечера', 'поздним вечером', 'к вечеру'],
    'ночь': ['ночью', 'ночи', 'глубокой ночью', 'за полночь']
}

y_true = []
y_pred = []

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

def classify_time_of_day(entity_text: str) -> str:
    for label, keywords in time_keywords.items():
        for keyword in keywords:
            if keyword in entity_text:
                return label
    return None

for _, row in df_marking.iterrows():
    true_label = row.get("time_of_day")

    if pd.isnull(true_label) or true_label not in time_mapping:
        continue

    text = get_full_text(row).lower()
    doc = nlp(text)

    predicted_label = None
    for ent in doc.ents:
        if ent.label_ == "TIME_OF_DAY":
            ent_text = ent.text.strip().lower()
            predicted_label = classify_time_of_day(ent_text)
            break

    if predicted_label in time_mapping:
        y_pred.append(time_mapping[predicted_label])
    else:
        y_pred.append(-1)

    y_true.append(time_mapping[true_label])

y_true_filtered = [yt for yp, yt in zip(y_pred, y_true) if yp != -1]
y_pred_filtered = [yp for yp in y_pred if yp != -1]

accuracy = accuracy_score(y_true_filtered, y_pred_filtered)
f1 = f1_score(y_true_filtered, y_pred_filtered, average="weighted")

print(f"Accuracy: {accuracy:.2%}")
print(f"F1-score: {f1:.2%}")
print(f"Пропущено {len(y_true) - len(y_true_filtered)} из {len(y_true)} примеров")

Accuracy: 97.96%
F1-score: 97.90%
Пропущено 33 из 82 примеров
